<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/banking77-intent-classification/notebooks/pretrained_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/TheFaridKarami/farid-ai-lab.git

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          DataCollatorWithPadding,
                          TrainingArguments,
                          Trainer,
                          set_seed)

SEED = 42
set_seed(SEED)

In [ ]:
PROJECT_ROOT = (Path("/content/farid-ai-lab")
 / "projects/nlp/banking77-intent-classification")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [ ]:
print(PROJECT_ROOT.exists())
print(DATA_DIR.exists())
print(TRAIN_PATH.exists())
print(TEST_PATH.exists())

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Shapes:", train_df.shape, test_df.shape)
print(train_df.dtypes)
print("Nulls:\n", train_df.isnull().sum())
print(train_df.head())

counts = train_df["label"].value_counts()
print("Classes train/test:", train_df["label"].nunique(), test_df["label"].nunique())
print(counts.describe())
print(counts.tail(5))
print("Missing from test:", set(train_df["label"]) - set(test_df["label"]))

print("Duplicate rows:", train_df.duplicated().sum())
print("Duplicate texts train/test:", train_df["text"].duplicated().sum(), test_df["text"].duplicated().sum())
print("Train/test text overlap:", len(set(train_df["text"]) & set(test_df["text"])))
print("Texts with conflicting labels:", (train_df.groupby("text")["label"].nunique() > 1).sum())

In [ ]:
train_df = pd.read_csv(TRAIN_PATH).drop_duplicates(subset="text").reset_index(drop=True)
test_df  = pd.read_csv(TEST_PATH)
assert len(train_df) == 10003, len(train_df)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    train_df["text"], train_df["label"],
    test_size=0.2, random_state=SEED, stratify=train_df["label"])

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(test_df["label"])

NUM_LABELS = len(le.classes_)
label2id = {str(l): int(i) for i, l in enumerate(le.classes_)}
id2label = {i: l for l, i in label2id.items()}

train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train.tolist()})
val_ds   = Dataset.from_dict({"text": X_val.tolist(),   "labels": y_val.tolist()})
test_ds  = Dataset.from_dict({"text": test_df["text"].tolist(), "labels": y_test.tolist()})
print(NUM_LABELS, len(train_ds), len(val_ds), len(test_ds))

In [ ]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = np.array([len(ids) for ids in tokenizer(X_train.tolist())["input_ids"]])

print(pd.Series(lengths).describe(percentiles=[.5, .9, .95, .99]))

MAX_LENGTH = 64
print("Coverage at MAX_LENGTH:", f"{(lengths <= MAX_LENGTH).mean().round(4)}%")

In [ ]:
def tokenize(batch):
  return tokenizer(batch["text"], max_length = MAX_LENGTH, truncation = True)

train_tok = train_ds.map(tokenize, batched = True, remove_columns = ["text"])
val_tok = val_ds.map(tokenize, batched = True, remove_columns = ["text"])
test_tok = test_ds.map(tokenize, batched = True, remove_columns = ["text"])

collector = DataCollectorWithPadding(tokenizer = tokenizer)

In [ ]:
def compute_metrics(eval_pred):
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis= -1)
   return {
       "Accuracy_score":accuracy_score(labels, predictions),
       "F1_score":f1_score(labels, predictions, average="macro")
   }

In [ ]:
model = AutoModelForSequenceClassfication(MODEL_NAME, num_labels = NUM_LABELS, label2id = label2id, id2label= id2label)